In [2]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# PARQUET_DIR = Path("parquet")

# con = duckdb.connect()


# files = sorted(PARQUET_DIR.rglob("*.parquet"))

# print(f"전체 파일: {len(files)}개")

# for f in files:
#     print(f)

In [ ]:
for file in files:

    print("\n" + "=" * 100)
    print(f"FILE: {file}")

    df_sample = con.execute(f"""
        SELECT *
        FROM read_parquet('{file}')
        LIMIT 10
    """).fetchdf()

    display(df_sample)

In [ ]:
# rename column name 

## ref - 3단계
# atcStep3Cd	atcStep3CdNm	diagYm	insupTpCd	msupUseAmt	st3SickSym	st3SickSymNm	totUseQty
# A02A	ANTACIDS	202201	4	20258	AE10	1형 당뇨병	549483

# 4단계ATC별상병별사용량목록조회 2020-2022/09_getAtcStp4SickList_2020.parquet
# 202001	1형 당뇨병	A01AD	4	16	17239	AE10	Other agents for local oral treatme
# diagYm - st3SickSymNm - atcStep4Cd- insupTpCd - msupUseAmt - totUseQty - st3SickSym - atcStep4CdNm
  
# 성분별상병별사용량목록조회 2020-2022/12_getCmpnSickList_2022.parquet
# 202201	1형 당뇨병	100701ACH	acebrophylline	4	424	86148	AE10
# diagYm - st3SickSymNm - gnlNmCd  - gnlNmCdNm - insupTpCd - msupUseAmt - totUseQty - st3SickSym

# concat same  data by diagYm

In [1]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

con.execute("""
CREATE OR REPLACE VIEW atc3_sick AS
SELECT * exclude(totUseQty,msupUseAmt) ,
msupUseAmt as totUseQty,
totUseQty as msupUseAmt,
FROM read_parquet(
    'parquet/atc3_sick/*.parquet',
    union_by_name = true
);

CREATE OR REPLACE VIEW mefi_sick AS
SELECT * exclude(totUseQty,msupUseAmt) ,
msupUseAmt as totUseQty,
totUseQty as msupUseAmt,
FROM read_parquet(
    'parquet/mefi_sick/*.parquet',
    union_by_name = true
);
""")

In [12]:
# DROP view IF EXISTS mefi_sick;
con.execute("""
select * from mefi_sick limit 1;

""").df()

,diagYm,meftDivNo,meftDivNoNm,st3SickSym,st3SickSymNm,insupTpCd,totUseQty,msupUseAmt
0,202001,111,전신마취제,AA03,시겔라증,4,2,4024


In [15]:
# con.execute("""SELECT table_schema, table_name, table_type
# FROM information_schema.tables
# WHERE table_name IN ('atc3_sick', 'mefi_sick');""").fetchall()

con.execute("""SELECT
    'atc3_sick' AS table_name,
    COUNT(*) AS row_count
FROM atc3_sick

UNION ALL

SELECT
    'mefi_sick' AS table_name,
    COUNT(*) AS row_count
FROM mefi_sick;""").fetchall()


[('atc3_sick', 17532309), ('mefi_sick', 12634246)]

In [3]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

con.execute("""

CREATE OR REPLACE VIEW atc4_sick AS

SELECT
    col_1::VARCHAR AS diagYm,
    col_2::VARCHAR AS st3SickSymNm,
    col_3::VARCHAR AS atcStep4Cd,
    col_4::VARCHAR AS insupTpCd,
    TRY_CAST(col_5 AS DOUBLE) AS totUseQty,
    TRY_CAST(col_6 AS DOUBLE) AS msupUseAmt,
    col_7::VARCHAR AS st3SickSym,
    col_8::VARCHAR AS atcStep4CdNm

FROM read_parquet(
    'parquet/atc4_sick/*.parquet',
    union_by_name = true
);

CREATE OR REPLACE VIEW cmpn_sick AS

SELECT
    col_1::VARCHAR AS diagYm,
    col_2::VARCHAR AS st3SickSymNm,
    col_3::VARCHAR AS gnlNmCd,
    col_4::VARCHAR AS gnlNmCdNm,
    col_5::VARCHAR AS insupTpCd,
    TRY_CAST(col_6 AS DOUBLE) AS totUseQty,
    TRY_CAST(col_7 AS DOUBLE) AS msupUseAmt,
    col_8::VARCHAR AS st3SickSym

FROM read_parquet(
    'parquet/cmpn_sick/*.parquet',
    union_by_name = true
);


""")

# healthcare.duckdb
# ├── VIEW atc4_sick
# ├── VIEW atc3_sick
# ├── VIEW cmpn_sick
# └── VIEW mefi_sick

con.execute("select * from atc4_sick limit 1;").df()

,diagYm,st3SickSymNm,atcStep4Cd,insupTpCd,totUseQty,msupUseAmt,st3SickSym,atcStep4CdNm
0,202001,1형 당뇨병,A01AD,4,16.0,17239.0,AE10,Other agents for local oral treatment


In [47]:
# con.execute("""CREATE OR REPLACE VIEW atc4_region as 
# SELECT
#     col_1::VARCHAR AS diagYm,
#     col_2::VARCHAR AS atcStep4Cd,
#     col_3::VARCHAR AS regionStep2Cd,
#     col_4::VARCHAR AS regionStep1CdNm,
#     col_5::VARCHAR AS regionStep2CdNm,
#     col_6::VARCHAR AS insupTpCd,
#     col_7::VARCHAR AS regionStep1Cd,
#     TRY_CAST(col_8 AS DOUBLE) AS totUseQty,
#     TRY_CAST(col_9 AS DOUBLE) AS msupUseAmt,
#     col_10::VARCHAR AS atcStep4CdNm    
#     FROM read_parquet(
#     'parquet/atc4_region/*.parquet',
#     union_by_name = true
# ) ; """)


con.execute("""CREATE OR REPLACE VIEW cmpn_region as 
SELECT
    col_1::VARCHAR AS diagYm,
     col_2::VARCHAR AS gnlNmCd,
    col_3::VARCHAR AS gnlNmCdNm,
     col_4::VARCHAR AS regionStep2Cd,
     col_5::VARCHAR AS regionStep2CdNm,
     col_6::VARCHAR AS regionStep1Cd,
     col_7::VARCHAR AS regionStep1CdNm,
     col_8::VARCHAR AS insupTpCd,     
     TRY_CAST(col_9 AS DOUBLE) AS totUseQty,
     TRY_CAST(col_10 AS DOUBLE) AS msupUseAmt
    FROM read_parquet(
    'parquet/cmpn_region/*.parquet',
    union_by_name = true
) ; """)

con.execute("select * from cmpn_region limit 1;").df()

,diagYm,gnlNmCd,gnlNmCdNm,regionStep2Cd,regionStep2CdNm,regionStep1Cd,regionStep1CdNm,insupTpCd,totUseQty,msupUseAmt
0,202001,100701ACH,acebrophylline,110001,강남구,110000,서울,4,45576.0,8986526.0


In [13]:
import duckdb

# con = duckdb.connect("healthcare.duckdb")

con.execute("""SELECT
*  
    FROM atc4_region
  limit 1; """).df()

,diagYm,atcStep4Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,atcStep4CdNm
0,202001,N05AX,210006,부산,부산서구,5,21,8333.0,3192425.0,Other antipsychotics


In [3]:
con.execute("""SELECT
*  
    FROM mefi_sick
  limit 1; """).df()

,diagYm,meftDivNo,meftDivNoNm,st3SickSym,st3SickSymNm,insupTpCd,totUseQty,msupUseAmt
0,202001,111,전신마취제,AA03,시겔라증,4,2,4024


In [ ]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

con.execute("""
CREATE OR REPLACE VIEW mefi_region as 
SELECT
    col_1::VARCHAR AS diagYm,
     col_2::VARCHAR AS meftDivNo,
    col_3::VARCHAR AS meftDivNoNm,
     col_4::VARCHAR AS regionStep2Cd,
     col_5::VARCHAR AS regionStep2CdNm,
     col_6::VARCHAR AS regionStep1Cd,
     col_7::VARCHAR AS regionStep1CdNm,
     col_8::VARCHAR AS insupTpCd,     
     TRY_CAST(col_9 AS DOUBLE) AS totUseQty,
     TRY_CAST(col_10 AS DOUBLE) AS msupUseAmt
    FROM read_parquet(
    'parquet/mefi_region/*.parquet',
    union_by_name = true
) 
; """).df()

con.execute("select * from mefi_region limit 1;").df()

,diagYm,meftDivNo,meftDivNoNm,regionStep2Cd,regionStep2CdNm,regionStep1Cd,regionStep1CdNm,insupTpCd,totUseQty,msupUseAmt
0,202001,111,전신마취제,110001,강남구,110000,서울,4,33860.0,198618276.0


In [11]:
# con.execute("""
# CREATE OR REPLACE VIEW mefi_region_inst  as
# SELECT
#     col_1::VARCHAR AS diagYm,
#      col_2::VARCHAR AS meftDivNo,
#     col_3::VARCHAR AS meftDivNoNm,
#      col_4::VARCHAR AS regionStep2Cd,
#      col_5::VARCHAR AS regionStep2CdNm,
#      col_6::VARCHAR AS regionStep1Cd,
#      col_7::VARCHAR AS regionStep1CdNm,
#      col_8::VARCHAR AS insupTpCd,     
#      TRY_CAST(col_9 AS DOUBLE) AS totUseQty,
#      TRY_CAST(col_10 AS DOUBLE) AS msupUseAmt,
#      col_11::VARCHAR AS medInstType
#     FROM read_parquet(
#     'parquet/mefi_hosp/*.parquet',
#     union_by_name = true
# ) 
# ; """).df()

con.execute("select * from mefi_region_inst limit 1;").df()



,diagYm,meftDivNo,meftDivNoNm,regionStep2Cd,regionStep2CdNm,regionStep1Cd,regionStep1CdNm,insupTpCd,totUseQty,msupUseAmt,medInstType
0,202001,399,따로 분류되지 않는 대사성 의약품,220004,인천중구,22,인천,4,93939.0,80796084.0,상급종합병원


In [12]:
con.execute("""
CREATE OR REPLACE VIEW atc4_region_inst  as
SELECT
    col_1::VARCHAR AS diagYm,
    col_2::VARCHAR AS atcStep4Cd,
    col_3::VARCHAR AS regionStep2Cd,
    col_4::VARCHAR AS regionStep1CdNm,
    col_5::VARCHAR AS regionStep2CdNm,
    col_6::VARCHAR AS insupTpCd,
    col_7::VARCHAR AS regionStep1Cd,
    TRY_CAST(col_8 AS DOUBLE) AS totUseQty,
    TRY_CAST(col_9 AS DOUBLE) AS msupUseAmt,
    col_10::VARCHAR AS medInstType,
    col_11::VARCHAR AS atcStep4CdNm    
     
    FROM read_parquet(
    'parquet/atc4_hosp/*.parquet',
    union_by_name = true
) 
; """).df()

con.execute("select * from atc4_region_inst limit 1;").df()



,diagYm,atcStep4Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,medInstType,atcStep4CdNm
0,202001,N05AX,210006,부산,부산서구,5,21,8333.0,3192425.0,병원,Other antipsychotics


In [10]:
con.execute("""
SELECT * EXCLUDE(st3SickSymNm,st3SickSym)
FROM atc4_sick WHERE atcStep4CdNm = 'Other antipsychotics'
 limit 1; """).df()

,diagYm,atcStep4Cd,insupTpCd,totUseQty,msupUseAmt,atcStep4CdNm
0,202001,N05AX,4,732.0,416717.0,Other antipsychotics


In [16]:
con.execute("""
CREATE OR REPLACE VIEW cmpn_region_inst  as
SELECT
    col_1::VARCHAR AS diagYm,
     col_2::VARCHAR AS gnlNmCd,
    col_3::VARCHAR AS gnlNmCdNm,
     col_4::VARCHAR AS regionStep2Cd,
     col_5::VARCHAR AS regionStep2CdNm,
     col_6::VARCHAR AS regionStep1Cd,
     col_7::VARCHAR AS regionStep1CdNm,
     col_8::VARCHAR AS insupTpCd,     
     TRY_CAST(col_9 AS DOUBLE) AS totUseQty,
     TRY_CAST(col_10 AS DOUBLE) AS msupUseAmt,
    col_11::VARCHAR AS medInstType
    FROM read_parquet(
    'parquet/cmpn_hosp/*.parquet',
    union_by_name = true
) 
; """).df()

con.execute("select * from cmpn_region_inst limit 1;").df()

,diagYm,gnlNmCd,gnlNmCdNm,regionStep2Cd,regionStep2CdNm,regionStep1Cd,regionStep1CdNm,insupTpCd,totUseQty,msupUseAmt,medInstType
0,202001,100701ACH,acebrophylline,110001,강남구,110000,서울,4,4751.0,494104.0,상급종합병원


### 신규테이블 점검

In [14]:
display(con.execute("describe  mefi_region_inst;").df())
display(con.execute("describe  mefi_region;").df())
display(con.execute("describe  atc4_region_inst;").df())

,column_name,column_type,null,key,default,extra
0,diagYm,VARCHAR,YES,None,None,None
1,meftDivNo,VARCHAR,YES,None,None,None
2,meftDivNoNm,VARCHAR,YES,None,None,None
3,regionStep2Cd,VARCHAR,YES,None,None,None
4,regionStep2CdNm,VARCHAR,YES,None,None,None
5,regionStep1Cd,VARCHAR,YES,None,None,None
6,regionStep1CdNm,VARCHAR,YES,None,None,None
7,insupTpCd,VARCHAR,YES,None,None,None
8,totUseQty,DOUBLE,YES,None,None,None
9,msupUseAmt,DOUBLE,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,diagYm,VARCHAR,YES,None,None,None
1,meftDivNo,VARCHAR,YES,None,None,None
2,meftDivNoNm,VARCHAR,YES,None,None,None
3,regionStep2Cd,VARCHAR,YES,None,None,None
4,regionStep2CdNm,VARCHAR,YES,None,None,None
5,regionStep1Cd,VARCHAR,YES,None,None,None
6,regionStep1CdNm,VARCHAR,YES,None,None,None
7,insupTpCd,VARCHAR,YES,None,None,None
8,totUseQty,DOUBLE,YES,None,None,None
9,msupUseAmt,DOUBLE,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,diagYm,VARCHAR,YES,None,None,None
1,atcStep4Cd,VARCHAR,YES,None,None,None
2,regionStep2Cd,VARCHAR,YES,None,None,None
3,regionStep1CdNm,VARCHAR,YES,None,None,None
4,regionStep2CdNm,VARCHAR,YES,None,None,None
5,insupTpCd,VARCHAR,YES,None,None,None
6,regionStep1Cd,VARCHAR,YES,None,None,None
7,totUseQty,DOUBLE,YES,None,None,None
8,msupUseAmt,DOUBLE,YES,None,None,None
9,medInstType,VARCHAR,YES,None,None,None


In [ ]:
for i

In [17]:

print(con.execute("""
SELECT 'atc4_sick' AS table_name, COUNT(*) AS rows
FROM atc4_sick
UNION ALL
SELECT 'atc3_sick', COUNT(*)
FROM atc3_sick
UNION ALL
SELECT 'cmpn_sick', COUNT(*)
FROM cmpn_sick
UNION ALL
SELECT 'mefi_sick', COUNT(*)
FROM mefi_sick
""").fetchall())

[('atc4_sick', 25374158), ('atc3_sick', 17532309), ('cmpn_sick', 47719163), ('mefi_sick', 12634246)]


In [35]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

con.execute("""
CREATE OR REPLACE VIEW region_medi_facil as
SELECT
*
FROM -- 탭 구분자(\t) 파일 조회하기
 read_csv('raw_data/region_medi_facility_20212022.tsv', 
 delim='\t')
  ; """)

con.execute("select * from region_medi_facil limit 1;").df()

,diagYm,sidoNm,population,pharm,advGenHosp,genHosp,hosp,longHosp,clinic,dentalClinic,orientalClinic
0,2021,서울특별시,"9,509,458","5,831",13,44,236,105,"10,323","4,903","3,662"


In [3]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

con.execute("""
CREATE OR REPLACE VIEW prod_atc_map as
SELECT
*,
    -- ATC 분리
    AtcCd[1:3] AS atc2cd,
    AtcCd[1:4] AS atc3cd,
    AtcCd[1:5] AS atc4cd,
    
    -- 1. 성분 일련번호 (함량/제형 상관없이 동일 성분인지 비교할 때 사용)
    gnlNmCd[1:4] AS cmpnId,
    
    -- 2. 함량 일련번호
    gnlNmCd[5:6] AS dsgId,
    
    -- 3. 투여경로 코드 및 직관적인 한글 명칭 매핑
    gnlNmCd[7] AS routeCd,
    CASE gnlNmCd[7]
        WHEN 'A' THEN '내복제'
        WHEN 'B' THEN '주사제'
        WHEN 'C' THEN '외용제'
        ELSE '기타/미분류'
    END AS routeNm,
    
    -- 4. 제형 코드 (정제, 캡슐, 액제 등 구분)
    gnlNmCd[8:9] AS formCd

FROM -- 탭 구분자(\t) 파일 조회하기
 read_csv('raw_data/cmpn_atc_map.txt', 
 delim='\t')
  ; """)

con.execute("select * from prod_atc_map limit 1;").df()

,meftDivNo,gnlNmCd,ProdCd,ProdNm,companyNm,AtcCd,AtcCdNm,atc2cd,atc3cd,atc4cd,cmpnId,dsgId,routeCd,routeNm,formCd
0,112,130830ASY,645302132,포크랄시럽(포수클로랄)_(9.5g/95mL),한림제약(주),N05CC01,chloral hydrate,N05,N05C,N05CC,1308,30,A,내복제,SY


In [6]:
con.execute("""
-- CREATE OR REPLACE VIEW atc_master as
SELECT
*
FROM 
 read_csv('raw_data/atc_master.csv')
limit 3
  ; """).df()

# con.execute("select * from prod_atc_map limit 1;").df()

,atc_code,atc_name,strength,uom,adm_r,note
0,A,ALIMENTARY TRACT AND METABOLISM,NA,NA,NA,NA
1,A01,STOMATOLOGICAL PREPARATIONS,NA,NA,NA,NA
2,A01A,STOMATOLOGICAL PREPARATIONS,NA,NA,NA,NA


In [ ]:
con.execute("").df().to_parquet('parquet/mefi_master.pq')
con.execute("").df().to_parquet('parquet/cmpn_master.pq')


# con.execute("select * from prod_atc_map limit 1;").df()

In [2]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

con.execute("""
CREATE OR REPLACE VIEW prod_atc_map_named AS
WITH mefi_master as (
    SELECT meftDivNo, meftDivNoNm FROM mefi_sick GROUP BY 1,2
),
cmpn_master as (
    SELECT gnlNmCd, gnlNmCdNm FROM cmpn_sick GROUP BY 1,2
),
atc4_master as (
    SELECT * FROM read_csv('raw_data/atc_master.csv') 
    WHERE length(atc_code) = 5
)
SELECT
    a.*,
    b.meftdivnoNm,
    c.gnlnmcdNm,
    d.atc_name as atc4CdNm
FROM prod_atc_map a
LEFT JOIN mefi_master b ON a.meftdivno = b.meftdivno
LEFT JOIN cmpn_master c ON a.gnlnmcd = c.gnlnmcd
LEFT JOIN atc4_master d ON a.ATC4Cd = d.atc_code;

SELECT * FROM prod_atc_map_named LIMIT 2;
""").df()

,meftDivNo,gnlNmCd,ProdCd,ProdNm,companyNm,AtcCd,AtcCdNm,atc2cd,atc3cd,atc4cd,cmpnId,dsgId,routeCd,routeNm,formCd,meftDivNoNm,gnlNmCdNm,atc4CdNm
0,123,221430BIJ,651202171,피리도민주(브롬화피리도스티그민)_(5mg/1mL),삼천당제약(주),N07AA02,pyridostigmine,N07,N07A,N07AA,2214,30,B,주사제,IJ,자율신경제,pyridostigmine bromide,Anticholinesterases
1,256,392430CCM,644802871,프라렉신크림(프라목신염산염)_(0.35g/35g),태극제약(주),C05AD07,pramocaine,C05,C05A,C05AD,3924,30,C,외용제,CM,치질용제,pramoxine hydrochloride,Local anesthetics


## total 점검

In [17]:
# con.execute("show tables;").df()
for tbname in [
    'mefi_sick','atc3_sick','atc4_sick','cmpn_sick', 
    'prod_atc_map',
    'mefi_region_inst','atc4_region_inst','cmpn_region_inst',
    'region_medi_facil'
]:
    # print(con.execute(f"""DESCRIBE {tbname};""").df())
    print("==== view name :  "+tbname)
    display(con.execute(f"""select * from {tbname} limit 3;""").df())


==== view name :  mefi_sick


,diagYm,meftDivNo,meftDivNoNm,st3SickSym,st3SickSymNm,insupTpCd,totUseQty,msupUseAmt
0,202001,111,전신마취제,AA03,시겔라증,4,2,4024
1,202001,111,전신마취제,AA04,기타 세균성 장감염,4,276,801611
2,202001,111,전신마취제,AA04,기타 세균성 장감염,5,13,27560


==== view name :  atc3_sick


,atcStep3Cd,atcStep3CdNm,diagYm,insupTpCd,st3SickSym,st3SickSymNm,totUseQty,msupUseAmt
0,A01A,STOMATOLOGICAL PREPARATIONS,202001,4,AE10,1형 당뇨병,16,17239
1,A01A,STOMATOLOGICAL PREPARATIONS,202001,5,AE10,1형 당뇨병,4,5000
2,A02A,ANTACIDS,202001,4,AE10,1형 당뇨병,21601,611090


==== view name :  atc4_sick


...

,diagYm,atcStep4Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,medInstType,atcStep4CdNm
0,202001,N05AX,210006,부산,부산서구,5,21,8333.0,3192425.0,병원,Other antipsychotics
1,202001,R01AD,210011,부산,부산금정구,5,21,10.0,119718.0,병원,Corticosteroids
2,202001,C07AG,210013,부산,부산연제구,5,31,435.0,231571.0,의원,Alpha and beta blocking agents


==== view name :  cmpn_region_inst


,diagYm,gnlNmCd,gnlNmCdNm,regionStep2Cd,regionStep2CdNm,regionStep1Cd,regionStep1CdNm,insupTpCd,totUseQty,msupUseAmt,medInstType
0,202001,100701ACH,acebrophylline,110001,강남구,110000,서울,4,4751.0,494104.0,상급종합병원
1,202001,100901ATB,aceclofenac,110001,강남구,110000,서울,4,11141.0,1157385.0,상급종합병원
2,202001,101401ATB,acetaminophen(encapsulated),110001,강남구,110000,서울,4,1683.0,43771.0,상급종합병원


==== view name :  region_medi_facil


,diagYm,sidoNm,population,pharm,advGenHosp,genHosp,hosp,longHosp,clinic,dentalClinic,orientalClinic
0,2021,서울특별시,"9,509,458","5,831",13,44,236,105,"10,323","4,903","3,662"
1,2021,부산광역시,"3,392,361","1,718",4,27,145,154,"2,682","1,353","1,133"
2,2021,대구광역시,"2,418,754","1,405",5,13,94,69,"2,049",955,901


In [53]:
# con.execute("show tables;").df()
for tbname in [
    'region_medi_facil','cmpn_region','cmpn_sick'
]:
    print("==== view name :  "+tbname)
    display(con.execute(f"""DESCRIBE {tbname};""").df())
    
    # print(con.execute(f"""select * from {tbname} limit 3;""").df())
#     print(con.execute(f"""SELECT DISTINCT insupTpCd
# FROM atc4_sick
# ORDER BY insupTpCd;""").df())
#     print(con.execute(f"""SELECT DISTINCT insupTpCd
# FROM atc4_region
# ORDER BY insupTpCd;""").df())


==== view name :  region_medi_facil


,column_name,column_type,null,key,default,extra
0,diagYm,BIGINT,YES,None,None,None
1,sidoNm,VARCHAR,YES,None,None,None
2,population,VARCHAR,YES,None,None,None
3,pharm,VARCHAR,YES,None,None,None
4,advGenHosp,BIGINT,YES,None,None,None
5,genHosp,BIGINT,YES,None,None,None
6,hosp,BIGINT,YES,None,None,None
7,longHosp,BIGINT,YES,None,None,None
8,clinic,VARCHAR,YES,None,None,None
9,dentalClinic,VARCHAR,YES,None,None,None


==== view name :  cmpn_region


,column_name,column_type,null,key,default,extra
0,diagYm,VARCHAR,YES,None,None,None
1,gnlNmCd,VARCHAR,YES,None,None,None
2,gnlNmCdNm,VARCHAR,YES,None,None,None
3,regionStep2Cd,VARCHAR,YES,None,None,None
4,regionStep2CdNm,VARCHAR,YES,None,None,None
5,regionStep1Cd,VARCHAR,YES,None,None,None
6,regionStep1CdNm,VARCHAR,YES,None,None,None
7,insupTpCd,VARCHAR,YES,None,None,None
8,totUseQty,DOUBLE,YES,None,None,None
9,msupUseAmt,DOUBLE,YES,None,None,None


==== view name :  cmpn_sick


,column_name,column_type,null,key,default,extra
0,diagYm,VARCHAR,YES,None,None,None
1,st3SickSymNm,VARCHAR,YES,None,None,None
2,gnlNmCd,VARCHAR,YES,None,None,None
3,gnlNmCdNm,VARCHAR,YES,None,None,None
4,insupTpCd,VARCHAR,YES,None,None,None
5,totUseQty,DOUBLE,YES,None,None,None
6,msupUseAmt,DOUBLE,YES,None,None,None
7,st3SickSym,VARCHAR,YES,None,None,None


## Quality Check

In [18]:
views_df = con.execute("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.views
    WHERE table_schema = 'main'
      AND table_name IN ('atc4_sick', 'atc3_sick', 'cmpn_sick', 'mefi_sick')
    ORDER BY table_name
""").fetchdf()

display(views_df)

,table_schema,table_name
0,main,atc3_sick
1,main,atc4_sick
2,main,cmpn_sick
3,main,mefi_sick


In [34]:
VIEWS = [
    "atc4_sick",
    "atc3_sick",
    "cmpn_sick",
    "mefi_sick",
]

row_counts = []

for view in VIEWS:
    n = con.execute(f"""
        SELECT COUNT(*)
        FROM "{view}"
    """).fetchone()[0]

    row_counts.append({
        "view": view,
        "row_count": n
    })

row_counts_df = pd.DataFrame(row_counts)

display(row_counts_df)

row_counts_dict = dict(
    zip(
        row_counts_df["view"],
        row_counts_df["row_count"]
    )
)

,view,row_count
0,atc4_sick,25374158
1,atc3_sick,17532309
2,cmpn_sick,47719163
3,mefi_sick,12634246


In [20]:
# ============================================================
# 3. VIEW 컬럼 구조
# ============================================================
import duckdb

con = duckdb.connect("healthcare.duckdb")

def get_columns(view):
    return con.execute(f"""
        SELECT
            ordinal_position,
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'main'
          AND table_name = '{view}'
        ORDER BY ordinal_position
    """).fetchdf()

VIEWS = [
    "atc4_sick",
    "atc3_sick",
    "cmpn_sick",
    "mefi_sick",
]

for view in VIEWS:

    print(f"\n{'=' * 80}")
    print(f"[{view}]")
    print("=" * 80)

    display(get_columns(view))


[atc4_sick]


,ordinal_position,column_name,data_type
0,1,diagYm,VARCHAR
1,2,st3SickSymNm,VARCHAR
2,3,atcStep4Cd,VARCHAR
3,4,insupTpCd,VARCHAR
4,5,msupUseAmt,DOUBLE
5,6,totUseQty,DOUBLE
6,7,st3SickSym,VARCHAR
7,8,atcStep4CdNm,VARCHAR



[atc3_sick]


,ordinal_position,column_name,data_type
0,1,atcStep3Cd,VARCHAR
1,2,atcStep3CdNm,VARCHAR
2,3,diagYm,VARCHAR
3,4,insupTpCd,VARCHAR
4,5,msupUseAmt,BIGINT
5,6,st3SickSym,VARCHAR
6,7,st3SickSymNm,VARCHAR
7,8,totUseQty,BIGINT



[cmpn_sick]


,ordinal_position,column_name,data_type
0,1,diagYm,VARCHAR
1,2,st3SickSymNm,VARCHAR
2,3,gnlNmCd,VARCHAR
3,4,gnlNmCdNm,VARCHAR
4,5,insupTpCd,VARCHAR
5,6,msupUseAmt,DOUBLE
6,7,totUseQty,DOUBLE
7,8,st3SickSym,VARCHAR



[mefi_sick]


,ordinal_position,column_name,data_type
0,1,diagYm,VARCHAR
1,2,meftDivNo,VARCHAR
2,3,meftDivNoNm,VARCHAR
3,4,st3SickSym,VARCHAR
4,5,st3SickSymNm,VARCHAR
5,6,insupTpCd,VARCHAR
6,7,msupUseAmt,BIGINT
7,8,totUseQty,BIGINT


In [33]:
def get_first_10rows(view):
    return con.execute(f"""
        SELECT
            diagYm, insupTpCd, st3SickSym, st3SickSymNm, totUseQty, msupUseAmt, 
            * EXCLUDE(diagYm, insupTpCd, st3SickSym, st3SickSymNm, totUseQty, msupUseAmt) 
        FROM '{view}'
        limit 10
    """).fetchdf()

for view in VIEWS:

    print(f"\n{'=' * 80}")
    print(f"[{view}]")
    print("=" * 80)

    display(get_first_10rows(view))


[atc4_sick]


,diagYm,insupTpCd,st3SickSym,st3SickSymNm,totUseQty,msupUseAmt,atcStep4Cd,atcStep4CdNm
0,202001,4,AE10,1형 당뇨병,17239.0,16.0,A01AD,Other agents for local oral treatment
1,202001,5,AE10,1형 당뇨병,5000.0,4.0,A01AD,Other agents for local oral treatment
2,202001,4,AE10,1형 당뇨병,309538.0,16697.0,A02AA,Magnesium compounds
3,202001,5,AE10,1형 당뇨병,168858.0,7701.0,A02AA,Magnesium compounds
4,202001,7,AE10,1형 당뇨병,9456.0,504.0,A02AA,Magnesium compounds
5,202001,4,AE10,1형 당뇨병,23890.0,806.0,A02AC,Calcium compounds
6,202001,5,AE10,1형 당뇨병,6150.0,205.0,A02AC,Calcium compounds
7,202001,4,AE10,1형 당뇨병,272748.0,4043.0,A02AD,"Combinations and complexes of aluminium, calci..."
8,202001,5,AE10,1형 당뇨병,238844.0,2824.0,A02AD,"Combinations and complexes of aluminium, calci..."
9,202001,7,AE10,1형 당뇨병,6360.0,60.0,A02AD,"Combinations and complexes of aluminium, calci..."



[atc3_sick]


,diagYm,insupTpCd,st3SickSym,st3SickSymNm,totUseQty,msupUseAmt,atcStep3Cd,atcStep3CdNm
0,202001,4,AE10,1형 당뇨병,17239,16,A01A,STOMATOLOGICAL PREPARATIONS
1,202001,5,AE10,1형 당뇨병,5000,4,A01A,STOMATOLOGICAL PREPARATIONS
2,202001,4,AE10,1형 당뇨병,611090,21601,A02A,ANTACIDS
3,202001,5,AE10,1형 당뇨병,416552,10820,A02A,ANTACIDS
4,202001,7,AE10,1형 당뇨병,15816,564,A02A,ANTACIDS
5,202001,4,AE10,1형 당뇨병,17827514,48364,A02B,DRUGS FOR PEPTIC ULCER AND GASTRO-OESOPHAGEAL ...
6,202001,5,AE10,1형 당뇨병,5441390,15369,A02B,DRUGS FOR PEPTIC ULCER AND GASTRO-OESOPHAGEAL ...
7,202001,7,AE10,1형 당뇨병,297307,870,A02B,DRUGS FOR PEPTIC ULCER AND GASTRO-OESOPHAGEAL ...
8,202001,4,AE10,1형 당뇨병,1083928,7314,A02X,
9,202001,5,AE10,1형 당뇨병,245428,2028,A02X,



[cmpn_sick]


,diagYm,insupTpCd,st3SickSym,st3SickSymNm,totUseQty,msupUseAmt,gnlNmCd,gnlNmCdNm
0,202001,4,AE10,1형 당뇨병,44676.0,216.0,100701ACH,acebrophylline
1,202001,5,AE10,1형 당뇨병,54180.0,258.0,100701ACH,acebrophylline
2,202001,7,AE10,1형 당뇨병,1050.0,5.0,100701ACH,acebrophylline
3,202001,4,AE10,1형 당뇨병,474169.0,2671.0,100901ATB,aceclofenac
4,202001,5,AE10,1형 당뇨병,136494.0,751.0,100901ATB,aceclofenac
5,202001,4,AE10,1형 당뇨병,52324.0,127.0,100903ATR,aceclofenac
6,202001,5,AE10,1형 당뇨병,12360.0,30.0,100903ATR,aceclofenac
7,202001,4,AE10,1형 당뇨병,4303.0,166.0,101401ATB,acetaminophen(encapsulated)
8,202001,4,AE10,1형 당뇨병,12826.0,444.0,101404ATB,acetaminophen(encapsulated)
9,202001,5,AE10,1형 당뇨병,2842.0,98.0,101404ATB,acetaminophen(encapsulated)



[mefi_sick]


,diagYm,insupTpCd,st3SickSym,st3SickSymNm,totUseQty,msupUseAmt,meftDivNo,meftDivNoNm
0,202001,4,AA03,시겔라증,4024,2,111,전신마취제
1,202001,4,AA04,기타 세균성 장감염,801611,276,111,전신마취제
2,202001,5,AA04,기타 세균성 장감염,27560,13,111,전신마취제
3,202001,4,AA05,달리 분류되지 않은 기타 세균성 음식매개중독,20293,7,111,전신마취제
4,202001,4,AA08,바이러스성 및 기타 명시된 장감염,220009,99,111,전신마취제
5,202001,5,AA08,바이러스성 및 기타 명시된 장감염,10381,6,111,전신마취제
6,202001,4,AA09,감염성 및 상세불명 기원의 기타 위장염 및 결장염,5251057,2441,111,전신마취제
7,202001,5,AA09,감염성 및 상세불명 기원의 기타 위장염 및 결장염,300452,136,111,전신마취제
8,202001,7,AA09,감염성 및 상세불명 기원의 기타 위장염 및 결장염,3733,2,111,전신마취제
9,202001,4,AA15,세균학적 및 조직학적으로 확인된 호흡기결핵,2529918,282,111,전신마취제


## qc
- null
- numeric : zero, negative
- datetime
- duplicate

In [ ]:
import pandas as pd


def null_quality(view_name):

    columns = get_columns(view_name)

    total_rows = con.execute(
        f'SELECT COUNT(*) FROM "{view_name}"'
    ).fetchone()[0]

    expressions = []

    for _, row in columns.iterrows():

        col = row["column_name"]
        dtype = str(row["data_type"]).upper()

        # column identifier 안전 처리
        qcol = '"' + col.replace('"', '""') + '"'

        # NULL
        expressions.append(f"""
            COUNT(*) FILTER (
                WHERE {qcol} IS NULL
            ) AS "{col}__null"
        """)

        # 문자열만 빈 문자열 검사
        if any(
            x in dtype
            for x in ["VARCHAR", "TEXT", "STRING"]
        ):
            expressions.append(f"""
                COUNT(*) FILTER (
                    WHERE {qcol} IS NOT NULL
                      AND TRIM(CAST({qcol} AS VARCHAR)) = ''
                ) AS "{col}__empty"
            """)

    sql = f"""
        SELECT
            {','.join(expressions)}
        FROM "{view_name}"
    """

    raw = con.execute(sql).fetchdf()

    results = []

    for _, row in columns.iterrows():

        col = row["column_name"]
        dtype = row["data_type"]

        null_count = int(raw.iloc[0][f"{col}__null"])

        empty_count = 0

        if any(
            x in str(dtype).upper()
            for x in ["VARCHAR", "TEXT", "STRING"]
        ):
            empty_count = int(
                raw.iloc[0][f"{col}__empty"]
            )

        results.append({
            "view": view_name,
            "column": col,
            "data_type": dtype,
            "null_count": null_count,
            "null_pct": round(
                null_count / max(total_rows, 1) * 100,
                4
            ),
            "empty_count": empty_count,
            "empty_pct": round(
                empty_count / max(total_rows, 1) * 100,
                4
            )
        })

    return pd.DataFrame(results)


null_results = []

for view in VIEWS:
    print(f"QC: {view}")
    null_results.append(null_quality(view))

null_df = pd.concat(
    null_results,
    ignore_index=True
)

display(
    null_df[
        (null_df["null_count"] > 0) |
        (null_df["empty_count"] > 0)
    ]
    .sort_values(
        ["view", "null_pct"],
        ascending=[True, False]
    )
)

# atcStep3CdNm, atcStep4CdNm, st3SickSymNm 은 사용하지 않는다. 다만 코드만 사용한다!

QC: atc4_sick
QC: atc3_sick
QC: cmpn_sick
QC: mefi_sick


,view,column,data_type,null_count,null_pct,empty_count,empty_pct
9,atc3_sick,atcStep3CdNm,VARCHAR,0,0.0000,577926,3.2963
14,atc3_sick,st3SickSymNm,VARCHAR,0,0.0000,108931,0.6213
7,atc4_sick,atcStep4CdNm,VARCHAR,517190,2.0383,0,0.0000
1,atc4_sick,st3SickSymNm,VARCHAR,131127,0.5168,0,0.0000
17,cmpn_sick,st3SickSymNm,VARCHAR,144414,0.3026,0,0.0000
28,mefi_sick,st3SickSymNm,VARCHAR,0,0.0000,94204,0.7456


In [24]:
def numeric_quality(view_name):

    columns = get_columns(view_name)

    numeric_columns = columns[
        columns["data_type"].str.upper().str.contains(
            "INTEGER|BIGINT|SMALLINT|TINYINT|HUGEINT|DECIMAL|DOUBLE|FLOAT"
        )
    ]

    if len(numeric_columns) == 0:
        return pd.DataFrame()

    expressions = []

    for _, row in numeric_columns.iterrows():

        col = row["column_name"]
        qcol = '"' + col.replace('"', '""') + '"'

        expressions.extend([
            f'MIN({qcol}) AS "{col}__min"',
            f'MAX({qcol}) AS "{col}__max"',
            f'AVG({qcol}) AS "{col}__avg"',

            f"""
            COUNT(*) FILTER (
                WHERE {qcol} < 0
            ) AS "{col}__negative"
            """,

            f"""
            COUNT(*) FILTER (
                WHERE {qcol} = 0
            ) AS "{col}__zero"
            """
        ])

    sql = f"""
        SELECT
            {','.join(expressions)}
        FROM "{view_name}"
    """

    raw = con.execute(sql).fetchdf()

    results = []

    for _, row in numeric_columns.iterrows():

        col = row["column_name"]

        results.append({
            "view": view_name,
            "column": col,
            "data_type": row["data_type"],
            "min": raw.iloc[0][f"{col}__min"],
            "max": raw.iloc[0][f"{col}__max"],
            "mean": raw.iloc[0][f"{col}__avg"],
            "negative_count": int(
                raw.iloc[0][f"{col}__negative"]
            ),
            "zero_count": int(
                raw.iloc[0][f"{col}__zero"]
            )
        })

    return pd.DataFrame(results)

numeric_results = []

for view in VIEWS:
    print(f"Numeric QC: {view}")

    result = numeric_quality(view)

    if not result.empty:
        numeric_results.append(result)

numeric_df = pd.concat(
    numeric_results,
    ignore_index=True
)

display(numeric_df)

Numeric QC: atc4_sick
Numeric QC: atc3_sick
Numeric QC: cmpn_sick
Numeric QC: mefi_sick


,view,column,data_type,min,max,mean,negative_count,zero_count
0,atc4_sick,msupUseAmt,DOUBLE,0.0,1.123388e+08,9.861909e+03,0,87141
1,atc4_sick,totUseQty,DOUBLE,0.0,4.492054e+10,4.453061e+06,0,1349
2,atc3_sick,msupUseAmt,BIGINT,0.0,1.512420e+08,1.500752e+04,0,56650
3,atc3_sick,totUseQty,BIGINT,0.0,6.164954e+10,6.846672e+06,0,1165
4,cmpn_sick,msupUseAmt,DOUBLE,0.0,4.683727e+07,3.653130e+03,0,244575
5,cmpn_sick,totUseQty,DOUBLE,0.0,1.183867e+10,1.353030e+06,0,5292
6,mefi_sick,msupUseAmt,BIGINT,0.0,2.588954e+08,2.120132e+04,0,37372
7,mefi_sick,totUseQty,BIGINT,0.0,9.652281e+10,9.597724e+06,0,6684


In [25]:
diagym_views = []

for view in VIEWS:

    columns = get_columns(view)["column_name"].tolist()

    if "diagYm" in columns:
        diagym_views.append(view)

print("diagYm 포함 VIEW:")
print(diagym_views)

for view in diagym_views:

    print(f"\n[{view}] invalid diagYm")

    result = con.execute(f"""
        SELECT
            diagYm,
            COUNT(*) AS row_count
        FROM "{view}"
        WHERE diagYm IS NOT NULL
          AND NOT regexp_matches(
              CAST(diagYm AS VARCHAR),
              '^[0-9]{{6}}$'
          )
        GROUP BY diagYm
        ORDER BY row_count DESC
    """).fetchdf()

    display(result)


diagYm 포함 VIEW:
['atc4_sick', 'atc3_sick', 'cmpn_sick', 'mefi_sick']

[atc4_sick] invalid diagYm


,diagYm,row_count



[atc3_sick] invalid diagYm


,diagYm,row_count



[cmpn_sick] invalid diagYm


,diagYm,row_count



[mefi_sick] invalid diagYm


,diagYm,row_count


In [26]:
date_range_df = []

for view in diagym_views:

    result = con.execute(f"""
        SELECT
            MIN(diagYm) AS min_diagYm,
            MAX(diagYm) AS max_diagYm,
            COUNT(DISTINCT diagYm) AS month_count
        FROM "{view}"
    """).fetchone()

    date_range_df.append({
        "view": view,
        "min_diagYm": result[0],
        "max_diagYm": result[1],
        "month_count": result[2]
    })

display(pd.DataFrame(date_range_df))

,view,min_diagYm,max_diagYm,month_count
0,atc4_sick,202001,202212,36
1,atc3_sick,202001,202212,36
2,cmpn_sick,202001,202212,36
3,mefi_sick,202001,202212,36


In [29]:
def check_full_duplicates(con, view_name):
    # 해당 VIEW의 컬럼 목록 조회
    columns = (
        con.execute(f"""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'main'
              AND table_name = '{view_name}'
            ORDER BY ordinal_position
        """)
        .fetchdf()["column_name"]
        .tolist()
    )

    if not columns:
        raise ValueError(f"컬럼을 찾을 수 없습니다: {view_name}")

    # 컬럼명을 안전하게 SQL identifier로 처리
    hash_args = ", ".join(
        f'"{col.replace(chr(34), chr(34) * 2)}"'
        for col in columns
    )

    sql = f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash({hash_args})) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash({hash_args})) AS duplicate_rows
        FROM "{view_name}"
    """


    return con.execute(sql).fetchdf()


duplicate_results = {}

for view in VIEWS:
    print(f"Checking: {view}")

    duplicate_results[view] = check_full_duplicates(con, view)

    display(
        duplicate_results[view].assign(view_name=view)
    )

Checking: atc4_sick


,total_rows,unique_rows,duplicate_rows,view_name
0,25374158,20635895,4738263,atc4_sick


Checking: atc3_sick


,total_rows,unique_rows,duplicate_rows,view_name
0,17532309,14364479,3167830,atc3_sick


Checking: cmpn_sick


,total_rows,unique_rows,duplicate_rows,view_name
0,47719163,37270844,10448319,cmpn_sick


Checking: mefi_sick


,total_rows,unique_rows,duplicate_rows,view_name
0,12634246,10326547,2307699,mefi_sick


In [36]:
def exact_duplicate_summary(con, view_name):
    columns = (
        con.execute(f"""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'main'
              AND table_name = '{view_name}'
            ORDER BY ordinal_position
        """)
        .fetchdf()["column_name"]
        .tolist()
    )

    if not columns:
        raise ValueError(f"컬럼을 찾을 수 없습니다: {view_name}")

    quoted_columns = ", ".join(
        f'"{c.replace(chr(34), chr(34) * 2)}"'
        for c in columns
    )

    sql = f"""
        WITH duplicate_groups AS (
            SELECT
                {quoted_columns},
                COUNT(*) AS group_count
            FROM "{view_name}"
            GROUP BY {quoted_columns}
            HAVING COUNT(*) > 1
        )
        SELECT
            COUNT(*) AS duplicate_groups,
            COALESCE(SUM(group_count - 1), 0) AS duplicate_rows,
            COALESCE(MAX(group_count), 0) AS max_repeat_count
        FROM duplicate_groups
    """

    return con.execute(sql).fetchdf()

for view in VIEWS:
    print(f"Checking exact duplicates: {view}")
    display(exact_duplicate_summary(con, view))

Checking exact duplicates: atc4_sick


: 

In [35]:
def exact_duplicate_summary(con, view_name):
    sql = f"""
        WITH duplicate_groups AS (
            SELECT
                COUNT(*) AS group_count
            FROM "{view_name}"
            GROUP BY ALL
            HAVING COUNT(*) > 1
        )
        SELECT
            COUNT(*) AS duplicate_groups,
            COALESCE(SUM(group_count - 1), 0) AS duplicate_rows,
            COALESCE(MAX(group_count), 0) AS max_repeat_count
        FROM duplicate_groups
    """

    return con.execute(sql).fetchdf()

for view in VIEWS:
    print(f"Checking exact duplicates: {view}")

    result = exact_duplicate_summary(con, view)
    display(result)

#     	view	row_count
# 0	atc4_sick	25374158
# 1	atc3_sick	17532309
# 2	cmpn_sick	47719163
# 3	mefi_sick	12634246


Checking exact duplicates: atc4_sick


,duplicate_groups,duplicate_rows,max_repeat_count
0,1,25374157.0,25374158


Checking exact duplicates: atc3_sick


,duplicate_groups,duplicate_rows,max_repeat_count
0,1,17532308.0,17532309


Checking exact duplicates: cmpn_sick


,duplicate_groups,duplicate_rows,max_repeat_count
0,1,47719162.0,47719163


Checking exact duplicates: mefi_sick


,duplicate_groups,duplicate_rows,max_repeat_count
0,1,12634245.0,12634246


In [ ]:
print("=" * 100)
print("DATA QUALITY SUMMARY")
print("=" * 100)

print("\n[1] ROW COUNT")
display(row_counts)

print("\n[2] NULL / EMPTY")
display(
    null_df[
        (null_df["null_count"] > 0) |
        (null_df["empty_count"] > 0)
    ]
    .sort_values(
        ["view", "null_pct"],
        ascending=[True, False]
    )
)

print("\n[3] NUMERIC")
display(numeric_df)

print("\n[4] DATE RANGE")
display(pd.DataFrame(date_range_df))

print("\n[5] DUPLICATE")
display(duplicate_df)

In [ ]:
 for file in files:
    schema = con.execute(f"""
        DESCRIBE SELECT *
        FROM read_parquet('{file}')
    """).fetchdf()

    display(schema)
 # 수치형 후보
    numeric_cols = []

    for col in schema["column_name"]:

        result = con.execute(f"""
            SELECT
                COUNT(*) AS n,
                COUNT(
                    TRY_CAST("{col}" AS DOUBLE)
                ) AS numeric_n
            FROM read_parquet('{file}')
        """).fetchone()

        n, numeric_n = result

        if n > 0 and numeric_n / n >= 0.95:
            numeric_cols.append(col)

    print("\n[NUMERIC COLUMNS]")
    print(numeric_cols)

    # 통계
    for col in numeric_cols:

        stats = con.execute(f"""
            SELECT
                MIN(TRY_CAST("{col}" AS DOUBLE)) AS min,
                QUANTILE_CONT(
                    TRY_CAST("{col}" AS DOUBLE), 0.25
                ) AS q1,
                MEDIAN(
                    TRY_CAST("{col}" AS DOUBLE)
                ) AS median,
                AVG(
                    TRY_CAST("{col}" AS DOUBLE)
                ) AS mean,
                QUANTILE_CONT(
                    TRY_CAST("{col}" AS DOUBLE), 0.75
                ) AS q3,
                MAX(
                    TRY_CAST("{col}" AS DOUBLE)
                ) AS max
            FROM read_parquet('{file}')
        """).fetchdf()

        print(f"\n[{col}]")
        display(stats)

        values = con.execute(f"""
            SELECT TRY_CAST("{col}" AS DOUBLE) AS value
            FROM read_parquet('{file}')
            WHERE TRY_CAST("{col}" AS DOUBLE) IS NOT NULL
        """).fetchdf()

        plt.figure(figsize=(9, 3))
        plt.boxplot(values["value"], vert=False)
        plt.title(f"{file.name} — {col}")
        plt.xlabel(col)
        plt.show()

date_range_df = []

for view in diagym_views:

    result = con.execute(f"""
        SELECT
            MIN(diagYm) AS min_diagYm,
            MAX(diagYm) AS max_diagYm,
            COUNT(DISTINCT diagYm) AS month_count
        FROM "{view}"
    """).fetchone()

    date_range_df.append({
        "view": view,
        "min_diagYm": result[0],
        "max_diagYm": result[1],
        "month_count": result[2]
    })

display(pd.DataFrame(date_range_df))

NameError: name 'files' is not defined

#### prod_atc_map

meftDivNo ↔ atc4cd가 실제로 1:N인지, N:N인지.

In [22]:
# con.execute("""
# select 
#   meftDivNo, count(distinct AtcCd)
# from prod_atc_map 
# group by 1
# having count(distinct MainCmpCd) >1
# limit 1;
# """).df()

display(
con.execute("""
SELECT
    atc4cd,
    COUNT(DISTINCT meftDivNo) AS mefi_count
FROM prod_atc_map
WHERE atc4cd IS NOT NULL
  AND meftDivNo IS NOT NULL
GROUP BY atc4cd
HAVING COUNT(DISTINCT meftDivNo) > 1
ORDER BY mefi_count DESC;;
""").df()

)

con.execute("""
SELECT
    AtcCd,
    COUNT(DISTINCT atc4cd) AS atc4_count
FROM prod_atc_map
WHERE AtcCd IS NOT NULL
  AND atc4cd IS NOT NULL
GROUP BY AtcCd
HAVING COUNT(DISTINCT atc4cd) > 1
ORDER BY atc4_count DESC;
""").df()



,atc4cd,mefi_count
0,,12
1,B05XA,6
2,L03AX,5
3,L04AX,5
4,C01CA,4
...,...,...
121,A05AA,2
122,H05BX,2
123,N02AX,2
124,J01XA,2


,AtcCd,atc4_count


In [23]:
display(
con.execute("""
SELECT
    COUNT(*) AS mapping_rows,
    COUNT(DISTINCT meftDivNo) AS mefi_count,
    COUNT(DISTINCT AtcCd) AS atc_count,
    COUNT(DISTINCT atc4cd) AS atc4_count
FROM prod_atc_map
WHERE meftDivNo IS NOT NULL
  AND AtcCd IS NOT NULL
  AND atc4cd IS NOT NULL;
""").df()

)

con.execute("""
SELECT
    atc4_count,
    COUNT(*) AS mefi_count
FROM (
    SELECT
        meftDivNo,
        COUNT(DISTINCT atc4cd) AS atc4_count
    FROM prod_atc_map
    WHERE meftDivNo IS NOT NULL
      AND atc4cd IS NOT NULL
    GROUP BY meftDivNo
)
GROUP BY atc4_count
ORDER BY atc4_count;
""").df()



,mapping_rows,mefi_count,atc_count,atc4_count
0,21952,124,1460,518


,atc4_count,mefi_count
0,1,29
1,2,18
2,3,14
3,4,12
4,5,7
5,6,14
6,7,4
7,8,5
8,9,2
9,10,2


In [24]:
con.execute("""
SELECT
    mefi_count,
    COUNT(*) AS atc4_count
FROM (
    SELECT
        atc4cd,
        COUNT(DISTINCT meftDivNo) AS mefi_count
    FROM prod_atc_map
    WHERE atc4cd IS NOT NULL
      AND meftDivNo IS NOT NULL
    GROUP BY atc4cd
)
GROUP BY mefi_count
ORDER BY mefi_count;""").df()

,mefi_count,atc4_count
0,1,392
1,2,94
2,3,20
3,4,8
4,5,2
5,6,1
6,12,1
